# Week 09 — A Spatial Data Warehouse

**Spatial Data Science with Python · MSc GIS / AGMS 1333**  
**Official Table 4 topic:** Data Warehousing · Fact tables, dimensions, SQLite, SQL

**Learning outcome.** Query date-sector summaries from a teaching warehouse.

**What you will implement:** SQLite, SQL joins and GROUP BY, population weighting.  
**Mathematical model:** Weighted mean = Σ(population × LST)/Σ observed population.

> **Data provenance:** The Urban Teaching Grid is a **synthetic** 12 × 12 grid at 250-m cell spacing. Its temperatures and facilities are simulated; do not present its numerical outputs as field measurements. All distance calculations in the labs use the projected coordinates EPSG:32647 where relevant.

**How to run.** In Colab, upload this notebook from *File → Upload notebook*. Run each cell top-to-bottom. The setup cell below will ask for `DATA_ONLY.zip` only if the data folder is absent. In JupyterLab, extract the complete ZIP and open this notebook in `01_Core_Labs/`.


## 1. Prepare the environment and locate the data

The same setup works in JupyterLab and Google Colab. It selects an existing local data directory, or prompts for the packaged dataset in Colab. `assert` fails early if the archive is incomplete.

In [ ]:
from pathlib import Path
import io
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Local Jupyter: data/ or ../data. Colab: upload DATA_ONLY.zip once per runtime.
candidates = [Path('data'), Path('../data'), Path('/content/data')]
DATA = next((p for p in candidates if (p / 'cells.csv').is_file()), None)
if DATA is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Cannot find data/. Extract the course ZIP and keep the data folder with the notebooks.') from exc
    print('Select DATA_ONLY.zip from the downloaded course package:')
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.endswith('.zip')), None)
    if zip_name is None:
        raise FileNotFoundError('Please upload DATA_ONLY.zip and run this cell again.')
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as archive:
        archive.extractall(Path.cwd())
    DATA = Path('data')
assert (DATA / 'metadata.json').is_file(), 'Data package incomplete.'
print('Dataset folder:', DATA.resolve())
print('Synthetic teaching data: not field observations.')


## 2. Key concepts and expected data

The analysis follows **input → cleaning → computation → validation → interpretation**. Always inspect observation keys, coordinate units, missing values, and the intended study period before fitting a model.

**Mathematical foundation:** Weighted mean = Σ(population × LST)/Σ observed population

**Method family:** SQLite, SQL joins and GROUP BY, population weighting.

## 3. Main computation

Run this cell after the setup. The block calculates the principal statistics or fits the weekly model. The original course code is retained so results remain aligned with the slide examples.

In [ ]:
import sqlite3
cells=pd.read_csv(DATA/'cells.csv'); obs=pd.read_csv(DATA/'observations_clean.csv')
conn=sqlite3.connect(':memory:')
cells[['cell_id','sector','population']].to_sql('dim_cell',conn,index=False,if_exists='replace')
obs.to_sql('fact_observation',conn,index=False,if_exists='replace')
print(pd.read_sql_query('SELECT COUNT(*) AS n_fact FROM fact_observation',conn))

### Reading the main code

**Warehouse preparation.** `sqlite3.connect(':memory:')` creates a temporary database; `to_sql` loads fact and dimension tables.

- `sqlite3` — Creates and queries a relational database from Python.
- `pd.read_csv` — Reads CSV text into a pandas DataFrame. Inspect head() and column types after loading.
- `SELECT ` — Selects columns or computed aggregates from SQL tables.

**Verification questions:** Which variables contain the inputs? What new object does the cell create? What numerical result should be checked before continuing?

## 4. Map, plot, or evaluate the result

The final block turns results into a labelled graphic or output artifact. Keep units in axes/legends, and distinguish model predictions from observations.

In [ ]:
sql="""SELECT d.sector,f.date,COUNT(*) AS expected_cells,COUNT(f.lst_c) AS observed_cells,
AVG(f.lst_c) AS unweighted_mean,
SUM(CASE WHEN f.lst_c IS NOT NULL THEN d.population*f.lst_c ELSE 0 END) /
NULLIF(SUM(CASE WHEN f.lst_c IS NOT NULL THEN d.population ELSE 0 END),0) AS weighted_mean
FROM fact_observation f JOIN dim_cell d ON f.cell_id=d.cell_id
GROUP BY d.sector,f.date ORDER BY f.date,d.sector"""
roll=pd.read_sql_query(sql,conn);print(roll.head())
roll.pivot(index='date',columns='sector',values='weighted_mean').plot(marker='o',title='Synthetic LST by sector and month')
plt.ylabel('Weighted LST (°C)');plt.show()

### Reading the output code

**SQL join.** `JOIN` adds sector and population attributes to each cell-date fact; `GROUP BY` aggregates by geography and date.

- `SELECT ` — Selects columns or computed aggregates from SQL tables.
- `plt.` — Creates a figure; title, axes, and legend are part of the analytical interpretation.

**Missing values.** `COUNT(*)` counts all rows, whereas `COUNT(lst_c)` counts non-null LST observations; weighted averages must exclude missing LST from both numerator and denominator.

## 5. Critical interpretation

### Interpret your results

Complete the following short report **after** running the notebook:

1. **Research question:** What geographic question does this notebook address?
2. **Observation unit and period:** What does one row/pixel represent? What is its date and CRS?
3. **Method:** Why is this method suitable, and what assumptions does it make?
4. **Results:** Report two numerical outputs or figure patterns with units.
5. **Validity:** State one data limitation and one model/evaluation limitation.
6. **Extension:** Modify one parameter, compare results, and describe what changed.


## 6. Questions and independent practice

1. Change the SQL query to return both weighted and unweighted sector means and their sample counts.
2. What would happen if the spatial unit, study date, or CRS changed?
3. Is this analysis descriptive, inferential, predictive, or a decision-support example? Justify your answer.
4. Which assumption should be tested before use in a real project?

**Next step:** Open the matching advanced notebook in `02_Advanced_Experiments/` for four short method variations.